# 01 — Dataset intake and structural validation

**Purpose:** establish an auditable baseline for the original CSV before changing, cleaning, or modeling it.

**Important:** this notebook does not overwrite the source file. Record the original filename and preserve an untouched copy.

## 1. Load the source dataset

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (78600, 14)


,timestamp,hour_of_day,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,2022-04-11 12:47:27,12,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,796.949206,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,2022-06-14 19:12:46,19,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,0.010000,purchase,South America,172.000,5,61,focused,established,25.00,low_risk
2,2022-01-18 16:26:59,16,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,778.197390,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,2022-06-15 09:20:04,9,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,300.838358,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,2022-02-18 14:35:30,14,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,775.569344,sale,Africa,172.160,6,100,high_value,veteran,62.50,moderate_risk


## 2. Schema, types, and missingness

In [2]:
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.astype(str).rename("dtype").to_frame())

profile = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_count": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(3),
    "n_unique": df.nunique(dropna=False)
})
display(profile.sort_values(["missing_count", "n_unique"], ascending=[False, True]))


Column names:
['timestamp', 'hour_of_day', 'sending_address', 'receiving_address', 'amount', 'transaction_type', 'location_region', 'ip_prefix', 'login_frequency', 'session_duration', 'purchase_pattern', 'age_group', 'risk_score', 'anomaly']

Data types:


,dtype
timestamp,object
hour_of_day,int64
sending_address,object
receiving_address,object
amount,float64
transaction_type,object
location_region,object
ip_prefix,float64
login_frequency,int64
session_duration,int64


,dtype,missing_count,missing_pct,n_unique
purchase_pattern,object,0,0.0,3
age_group,object,0,0.0,3
anomaly,object,0,0.0,3
transaction_type,object,0,0.0,5
location_region,object,0,0.0,5
ip_prefix,float64,0,0.0,5
login_frequency,int64,0,0.0,8
hour_of_day,int64,0,0.0,24
risk_score,float64,0,0.0,31
session_duration,int64,0,0.0,140


## 3. Duplicates, constants, and suspicious representations

In [3]:
print("Exact duplicate rows:", int(df.duplicated().sum()))
print("Constant columns:", [c for c in df.columns if df[c].nunique(dropna=False) <= 1])

for c in df.select_dtypes(include=["object"]).columns:
    print(f"\\n{c}: {df[c].nunique(dropna=False)} unique values")
    display(df[c].value_counts(dropna=False).head(20).to_frame("count"))

Exact duplicate rows: 0
Constant columns: []
\ntimestamp: 78513 unique values


,count
timestamp,
2022-03-28 05:26:31,2
2022-06-01 06:46:51,2
2022-04-10 05:30:51,2
2022-04-15 09:06:17,2
2022-12-27 13:39:10,2
2022-08-02 18:21:44,2
2022-05-22 08:13:52,2
2022-02-06 17:59:18,2
2022-08-27 02:49:19,2


\nsending_address: 1161 unique values


,count
sending_address,
0x4bfe6fa35149db081520aea5e503946572718756,232
0xb0b2f6fc707fbb7f9d27a9f4fe0cb0d6b39a0155,204
0xccbf5a9926606c77db8391185066a8ee3bc0a2af,203
0xf24dd92ee52bb0de1bab87766a4a3f1c17b2c4d9,203
0x91ee0c3b3f87d070e260f6208c52534d36817dae,203
0x1d132c165feddfa72ab5fca8b7dfa522efb53a56,203
0xceda6c769350a084f02a6e5f2fadf269ecc0c37d,203
0x09802f792bfbbf1f8515808f0fa68344c5625dbc,174
0x5b6124ac76b58955e1de4c58d87e5bca4c881aee,174


\nreceiving_address: 1166 unique values


,count
receiving_address,
0x86b61ffe9ce4da504e6b7cf3f0925c1941265074,232
0x0496b829a3ede37a92aa2b9b101549ed23018cb8,203
0x897c07cac7f9149d0be00903c55d810f6e42c9cc,203
0x2c3cfb5537f82b3baf41b0869f20b66f47d57c23,203
0x617b83371b4c65debb6c2e90de89c827ddcb0884,203
0x90c61e65de8543a1e3daeda46d03fea897886666,203
0x4e3bf6cc6fa25ce2b65d184fc14d52f375768e3c,203
0xadc445c80282504e4af6afeb5681fded2c3d6069,203
0x880517c4e61f634eb26e59e702726e2cbae53959,203


... (6 outputs left out)

\nage_group: 3 unique values


,count
age_group,
veteran,26422
new,26145
established,26033


\nanomaly: 3 unique values


,count
anomaly,
low_risk,63494
moderate_risk,8611
high_risk,6495


## 4. Timestamp and derived-time consistency

In [4]:
if "timestamp" in df.columns:
    ts = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
    print("Unparseable timestamps:", int(ts.isna().sum()))
    print("Timestamp min/max:", ts.min(), ts.max())
    if "hour_of_day" in df.columns:
        mismatch = ts.notna() & (df["hour_of_day"].astype("Int64") != ts.dt.hour.astype("Int64"))
        print("hour_of_day mismatches:", int(mismatch.sum()))

Unparseable timestamps: 0
Timestamp min/max: 2022-01-01 00:00:45+00:00 2022-12-31 23:47:40+00:00
hour_of_day mismatches: 0


## 5. Numeric distributions and category summaries

In [5]:
display(df.describe(include="all").T)

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
timestamp,78600,78513,2022-03-28 05:26:31,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hour_of_day,78600.0,NaN,NaN,NaN,11.532634,6.935897,0.0,6.0,12.0,18.0,23.0
sending_address,78600,1161,0x4bfe6fa35149db081520aea5e503946572718756,232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
receiving_address,78600,1166,0x86b61ffe9ce4da504e6b7cf3f0925c1941265074,232,NaN,NaN,NaN,NaN,NaN,NaN,NaN
amount,78600.0,NaN,NaN,NaN,502.574903,245.898146,0.01,331.319966,500.0295,669.528311,1557.150905
transaction_type,78600,5,sale,25040,NaN,NaN,NaN,NaN,NaN,NaN,NaN
location_region,78600,5,North America,15840,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ip_prefix,78600.0,NaN,NaN,NaN,147.64443,69.388143,10.0,172.0,172.16,192.0,192.168
login_frequency,78600.0,NaN,NaN,NaN,4.178702,2.366038,1.0,2.0,4.0,6.0,8.0
session_duration,78600.0,NaN,NaN,NaN,69.684606,40.524476,20.0,35.0,60.0,100.0,159.0


## 6. Export a compact audit report

In [6]:
report_path = Path("01_dataset_intake_profile.csv")
profile.to_csv(report_path)
print("Saved:", report_path.resolve())

Saved: /content/01_dataset_intake_profile.csv


## Interpretation checklist

- Confirm that row count and columns match the documented source version.
- Explain every missing value, duplicate, constant, and unusual data type.
- Check whether derived fields can be reconstructed from raw fields.
- Do not drop columns or rows in this notebook; document decisions first.